# 1. Setup

Install packages

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

# Dataset retrieval

Dataset source: https://huggingface.co/datasets/austindavis/lichess-uci

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

ds = load_dataset("austindavis/lichess-uci", "201505")["train"]

# First cut: split off test set — this is quarantined until the end
train_val = ds.train_test_split(test_size=0.05, seed=42)

# Second cut: split train into train and validation
train_val_split = train_val["train"].train_test_split(test_size=0.05, seed=42)

final_splits = {
    "train": train_val_split["train"],
    "val":   train_val_split["test"],
    "test":  train_val["test"],
}

ds = DatasetDict(final_splits)
ds

In [ ]:
ds["train"][0]

In [ ]:
RESULT_STRINGS = {"1-0", "0-1", "1/2-1/2", "*"}

def clean_transcript(sample):
    cleaned_transcripts = []
    cleaned_moves = []
    
    for transcript in sample["Transcript"]:
        # Remove result string from end if present
        moves = transcript.lower().split()
        if moves and moves[-1] in RESULT_STRINGS:
            moves = moves[:-1]
        cleaned_transcripts.append(" ".join(moves))
        cleaned_moves.append(moves)
    
    return {
        "Transcript": cleaned_transcripts,
        "Moves": cleaned_moves
    }

ds = ds.map(clean_transcript, batched=True, num_proc=2, desc="Cleaning transcripts")
ds["train"][0]

# Tokenization

## Vocabular-ization

There are a small number of legal moves in UCI notation. Therefore, we can construct a small vocabulary:

Now, we build actual tokens

In [ ]:
from collections import Counter

# Special tokens first, then all moves sorted by frequency
# Sorting by frequency is a minor nicety — most common tokens
# get low IDs, which can matter for some embedding implementations
SPECIAL_TOKENS = [
    "[PAD]",        # 0 - padding (you may not need this with flat binary approach)
    "[START]",      # 1 - beginning of game
    "[CHECKMATE]",  # 2
    "[STALEMATE]",  # 3
    "[DRAW_REP]",   # 4 - threefold/fivefold repetition
    "[DRAW_50]",    # 5 - 50/75 move rule
    "[DRAW_MAT]",   # 6 - insufficient material
    "[RESIGN]",     # 7 - resignation
    "[TIMEOUT]",    # 8 - TIMEOUT
    "[UNK]",        # 9 - unknown/malformed move (safety net)
]

move_counter = Counter()
for moves in ds["train"]["Moves"]:
    move_counter.update(moves)

print(f"Unique moves after cleaning: {len(move_counter)}")
# Should be ~1966, and 1-0/0-1/1/2-1/2 should be gone
print(f"1-0 still in counter: {'1-0' in move_counter}")


# Build vocab: special tokens + all moves by descending frequency
all_moves_sorted = [move for move, _ in move_counter.most_common()]
vocab = SPECIAL_TOKENS + all_moves_sorted

token2id = {token: idx for idx, token in enumerate(vocab)}
id2token = {idx: token for token, idx in token2id.items()}

len(vocab)

Build a token-to-id mapper, and inverse. We also save the vocab for convenience.

Token test

In [ ]:
# Round-trip test — should print True
test_token = "e2e4"
assert id2token[token2id[test_token]] == test_token
print(f"vocab size: {len(vocab)}")
print(f"e2e4 is token {token2id['e2e4']}")
print(f"token 1 is {id2token[1]}")  # should be [START]
print(f"rarest move: {vocab[-1]} at index {len(vocab)-1}")

In [ ]:
from collections import Counter

term_result_counter = Counter(
    zip(ds["train"]["Termination"], ds["train"]["Result"])
)

for (term, result), count in term_result_counter.most_common():
    print(f"({term!r}, {result!r}): {count:,}")

In [ ]:
# mass tokenize games taking advantage of ds.map() again

from typing import Any

TERM_MAP = {
    ("Normal",           "1-0"):     "[RESIGN]",
    ("Normal",           "0-1"):     "[RESIGN]",
    ("Normal",           "1/2-1/2"): "[DRAW_REP]",
    ("Time forfeit",     "1-0"):     "[TIMEOUT]",
    ("Time forfeit",     "0-1"):     "[TIMEOUT]",
    ("Time forfeit",     "1/2-1/2"): "[TIMEOUT]",
    ("Abandoned",        "1-0"):     "[RESIGN]",
    ("Abandoned",        "0-1"):     "[RESIGN]",
    ("Abandoned",        "1/2-1/2"): "[DRAW_REP]",
    ("Rules infraction", "1-0"):     "[RESIGN]",
}

import numpy as np

def make_tokenize_fn(token2id, TERM_MAP):
    START_ID = token2id["[START]"]
    UNK_ID   = token2id["[UNK]"]
    
    def tokenize_batch(batch):
        all_encoded = []
        
        for moves, termination, result in zip(
            batch["Moves"],
            batch["Termination"],
            batch["Result"]
        ):
            end_token = TERM_MAP.get((termination, result))
            
            if not moves or result == "*" or end_token is None:
                all_encoded.append(None)
                continue
            
            ids = [START_ID]
            for move in moves:
                ids.append(token2id.get(move, UNK_ID))
            ids.append(token2id[end_token])
            
            all_encoded.append(ids)
        
        return {"input_ids": all_encoded}
    
    return tokenize_batch

tokenize_fn = make_tokenize_fn(token2id, TERM_MAP)

ds = ds.map(
    tokenize_fn,
    batched=True,
    batch_size=10_000,
    num_proc=2,       # Kaggle gives you multiple cores
    desc="Tokenizing"
)

In [ ]:
def write_binary(split, out_path):
    all_ids = []
    skipped = 0
    
    for row in split:
        ids = row["input_ids"]
        if ids is None:
            skipped += 1
            continue
        all_ids.extend(ids)
    
    arr = np.array(all_ids, dtype=np.uint16)
    arr.tofile(out_path)
    print(f"{out_path}: {len(arr):,} tokens, "
          f"{arr.nbytes/1e6:.1f} MB, "
          f"{skipped:,} skipped")


write_binary(ds["train"], "./bin/train_tokens.bin")
write_binary(ds["val"],   "./bin/val_tokens.bin")
write_binary(ds["test"],  "./bin/test_tokens.bin")

In [ ]:
import numpy as np

for name, path in [
    ("train", "./bin/train_tokens.bin"),
    ("val",   "./bin/val_tokens.bin"),
    ("test",  "./bin/test_tokens.bin"),
]:
    arr = np.fromfile(path, dtype=np.uint16)
    n_starts = int((arr == token2id["[START]"]).sum())
    print(f"{name}: {len(arr):,} tokens, {n_starts:,} games, "
          f"avg {len(arr)/n_starts:.1f} tokens/game")

In [ ]:
train = np.fromfile("./bin/train_tokens.bin", dtype=np.uint16)
start_positions = np.where(train == token2id["[START]"])[0]

print("First 3 games decoded:")
for i in range(3):
    start = start_positions[i]
    end   = start_positions[i+1] if i+1 < len(start_positions) else len(train)
    tokens = train[start:end]
    decoded = [id2token[int(t)] for t in tokens]
    print(f"\nGame {i+1} ({len(decoded)} tokens):")
    print(" ".join(decoded))

## Tokenizer

We now build the actual tokenizer.

We use the WordLevel Tokenizer since every of our moves (like `e2e4`) need to be recorded in full, as well as the fact that moves are separated by spaces.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordLevel(vocab=token2id, unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

tokenizer.add_special_tokens(SPECIAL_TOKENS)

tokenizer.save("./tokenizer/tokenizer.json")
print("Tokenizer saved.")

Tokenizer 

In [ ]:
enc = tokenizer.encode("e2e4 e7e5 g1f3 b8c6")
print(enc.tokens)   # ['e2e4', 'e7e5', 'g1f3', 'b8c6']
print(enc.ids)      # [some integers matching your token2id]

# Round trip
decoded = tokenizer.decode(enc.ids)
print(decoded)      # 'e2e4 e7e5 g1f3 b8c6'

In [ ]:
# Encode a game prefix to feed to the model
moves_so_far = "e2e4 e7e5 g1f3"
ids = tokenizer.encode(moves_so_far).ids
# prepend [START] manually since encode() doesn't add it automatically
ids = [token2id["[START]"]] + ids
print(ids)

# Decode model output back to moves
predicted_id = 423  # whatever the model sampled
predicted_move = tokenizer.id_to_token(predicted_id)  # "b8c6"
print(predicted_move)

In [ ]:
print("Special Tokens:", [token for token in token2id if token.startswith("[")])

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size          = len(vocab),      # 1980
    n_positions         = 256,             # max sequence length
    n_embd              = 256,             # d_model
    n_layer             = 6,               # transformer blocks
    n_head              = 8,               # attention heads
    n_inner             = 1024,            # MLP hidden dim (4 * n_embd)
    activation_function = "gelu_new",
    resid_pdrop         = 0.1,
    embd_pdrop          = 0.1,
    attn_pdrop          = 0.1,
    bos_token_id        = token2id["[START]"],
    eos_token_id        = [
        token2id["[RESIGN]"],
        token2id["[TIMEOUT]"],
        token2id["[DRAW_REP]"],
        token2id["[CHECKMATE]"],
        token2id["[STALEMATE]"],
        token2id["[DRAW_MAT]"],
        token2id["[DRAW_50]"],
    ],
    attn_implementation = "sdpa",          # the one good part from that line
)

# Randomly initialized — no pretrained weights
model = GPT2LMHeadModel(config)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
# Should be ~8-10M